# 🌙 UNLET-ADAS: Low-Light Detector Training (Colab, free GPU)
### B.E. Major Project | SJBIT Bengaluru | CSE 2025-26
**GitHub:** https://github.com/DEEK-SHITH/UNLET-ADAS

Fine-tunes a dedicated **5-class low-light specialist** YOLOv8 model
(Person / Bicycle / Car / Motorcycle / Bus) on ExDark (Exclusively
Dark Image Dataset — Loh & Chan, CVIU 2019), so detection is trained
on real night-time appearance instead of relying entirely on stock
COCO daylight weights run on enhanced frames.

This is a **separate model**, not a replacement for the main 8-class
ADAS detector: ExDark has no Truck / Traffic Light / Stop Sign data,
and fine-tuning directly on its 12 classes would silently drop those
3 classes rather than leave them unchanged (Ultralytics resizes the
detection head to match the training class list). The app offers this
as an alternative "Detector Model" choice alongside the stock COCO
model, which keeps full 8-class daylight coverage.

**Dataset sourcing — read this before Cell 2:** ExDark's official
release isn't a ready-made YOLO export, and several independent
YOLO-format mirrors of varying completeness exist on Roboflow
Universe — there's no single verified one to hardcode, so **you must
find one yourself**:
1. Go to https://universe.roboflow.com and search **"ExDark"**
2. Pick a project whose image count is close to the real dataset's
   **~7,363 images** — a much smaller count usually means a partial
   subset (e.g. person-only)
3. Click **Download Dataset → YOLOv8 → Show download code** and read
   off the workspace / project / version from the generated
   `rf.workspace(...).project(...).version(...)` snippet
4. Paste those three values into Cell 2 below

| Cell | What it does |
|---|---|
| 1 | Setup — install packages, mount Drive, clone GitHub |
| 2 | Configuration — Roboflow key/workspace/project/version + paths |
| 3 | Download + filter the dataset to the 5 ADAS-relevant classes |
| 4 | Train YOLOv8 low-light detector (~20–35 min on a T4 GPU) |
| 5 | Check results — metrics + sample predictions |
| 6 | Save weights + deployment instructions |

**Run cells top to bottom. Do not skip any cell.**

You need a free Roboflow API key for Cell 2 — sign up at
https://app.roboflow.com, then **Settings → API Keys**. Paste your own
key there; never share it publicly (if a key you pasted somewhere ever
becomes visible to others, regenerate it from that same Settings page).


In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================

# Anti-disconnect — run this first
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    var btns = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<btns.length;i++){
        if(btns[i].id=="connect") btns[i].click();
    }
}
setInterval(ClickConnect, 55000)
'''))
print('Anti-disconnect active!')

# Mount Google Drive (so trained weights survive when the Colab runtime recycles)
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install ultralytics roboflow -q

# Clone or update GitHub repo
import os
if not os.path.exists('/content/UNLET-ADAS'):
    !git clone https://github.com/DEEK-SHITH/UNLET-ADAS.git /content/UNLET-ADAS
    print('Repo cloned!')
else:
    !cd /content/UNLET-ADAS && git pull
    print('Repo updated!')

import sys
if '/content/UNLET-ADAS' not in sys.path:
    sys.path.insert(0, '/content/UNLET-ADAS')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
print('Setup complete!')


In [ ]:
# ============================================================
# CELL 2 — Configuration
# All paths (and your Roboflow key/workspace/project/version) are
# defined here. Edit only this cell if settings need to change.
# ============================================================

ROBOFLOW_API_KEY       = 'PASTE_YOUR_OWN_KEY_HERE'   # app.roboflow.com -> Settings -> API Keys
ROBOFLOW_WORKSPACE     = 'PASTE_WORKSPACE_HERE'       # from the "Show download code" snippet
ROBOFLOW_PROJECT       = 'PASTE_PROJECT_HERE'         # from the "Show download code" snippet
ROBOFLOW_VERSION       = 1                            # from the "Show download code" snippet

DATASET_DIR = '/content/lowlight_dataset'                             # downloaded fresh each run
SAVE_DIR    = '/content/drive/MyDrive/UNLET_Project/checkpoints'      # persists across sessions
MODEL_SIZE  = 'n'          # n = fastest/smallest, matches the lightweight theme of this project
EPOCHS      = 60
BATCH_SIZE  = 16
IMAGE_SIZE  = 640
PATIENCE    = 15           # early stop if val mAP doesn't improve for this many epochs

os.makedirs(SAVE_DIR, exist_ok=True)

_missing = [n for n, v in [
    ('ROBOFLOW_API_KEY', ROBOFLOW_API_KEY),
    ('ROBOFLOW_WORKSPACE', ROBOFLOW_WORKSPACE),
    ('ROBOFLOW_PROJECT', ROBOFLOW_PROJECT),
] if v.startswith('PASTE_')]
if _missing:
    print(f'MISSING: set {", ".join(_missing)} above before continuing.')
else:
    print('Configuration OK. Ready for Cell 3.')
print(f'Save dir : {SAVE_DIR}')


In [ ]:
# ============================================================
# CELL 3 — Download the ExDark dataset from Roboflow, then filter it
# down to the 5 ADAS-relevant classes (Person/Bicycle/Car/Motorcycle/
# Bus), remapped to ids 0..4. Watch the printed box-count report --
# if any of the 5 classes shows 0 boxes, this Roboflow mirror is
# missing data for it; go back to Cell 2 and try a different project.
# ============================================================

from src.train_lowlight import download_dataset, filter_and_remap_dataset

print('Downloading ExDark dataset from Roboflow...')
raw_location = download_dataset(
    ROBOFLOW_API_KEY, DATASET_DIR,
    ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION)
print(f'Raw dataset at: {raw_location}')

print('\nFiltering to ADAS-relevant classes and remapping IDs...')
filtered_dir = os.path.join(DATASET_DIR, 'filtered')
data_yaml = filter_and_remap_dataset(raw_location, filtered_dir)

print(f'\nFiltered dataset ready at: {filtered_dir}')
print(f'data.yaml                : {data_yaml}')
assert os.path.exists(data_yaml), 'data.yaml not found — check the output above for errors.'


In [ ]:
# ============================================================
# CELL 4 — Train YOLOv8 low-light detector (~20-35 min on a T4 GPU)
# ============================================================

from ultralytics import YOLO

model = YOLO(f'yolov8{MODEL_SIZE}.pt')
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    project=SAVE_DIR,
    name='lowlight_run',
    exist_ok=True,
)

print('\nTraining complete!')


In [ ]:
# ============================================================
# CELL 5 — Check results: metrics + sample predictions
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = os.path.join(SAVE_DIR, 'lowlight_run')

# Training curves (loss / mAP / precision / recall over epochs)
results_png = os.path.join(run_dir, 'results.png')
if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training curves')
    plt.show()

# Sample validation predictions (model's own boxes drawn on val images)
val_pred = os.path.join(run_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_pred):
    img = mpimg.imread(val_pred)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Sample validation predictions')
    plt.show()

best_weights = os.path.join(run_dir, 'weights', 'best.pt')
print(f'\nBest weights: {best_weights}')
print(f'Exists      : {os.path.exists(best_weights)}')


In [ ]:
# ============================================================
# CELL 6 — Save weights + deployment instructions
# ============================================================

import shutil

best_weights = os.path.join(SAVE_DIR, 'lowlight_run', 'weights', 'best.pt')
final_path   = os.path.join(SAVE_DIR, 'lowlight_best.pt')

if os.path.exists(best_weights):
    shutil.copy(best_weights, final_path)
    print(f'Saved to Google Drive: {final_path}')
    print()
    print('To enable the Low-Light Detector option in the Streamlit app:')
    print('  1. Download this file from your Google Drive.')
    print('  2. Rename it to: yolov8_lowlight.pt')
    print('  3. Place it at: app/yolov8_lowlight.pt   (in your local UNLET-ADAS clone)')
    print('  4. Restart the Streamlit app -- the "Detector Model" dropdown will gain a')
    print('     "yolov8_lowlight.pt (fine-tuned, ExDark)" option. Note it only detects')
    print('     Person/Bicycle/Car/Motorcycle/Bus -- no Traffic Light/Stop Sign/Truck.')
else:
    print(f'Expected weights at {best_weights} but they were not found -- '
          'check Cell 4\'s training log for errors.')
